# IDRiD Disease Grading Testing Set — one-time external evaluation of the frozen Stage 1–8 pipeline (record §60)

**Evaluation only. Nothing is trained, tuned or selected.** Everything that defines the run is in `idrid_grading_protocol.json`:
the 103 test images and labels (hashes), the three images excluded from the primary set, the eight model files (SHA-256),
the inference settings, the locked fusion rule and the parity tolerances.

Order of the cells — it cannot be changed:
1. Setup.
2. Protocol and checkpoint verification.
3. **APTOS parity gate.** Ten authoritative APTOS validation images go through this exact code from the raw image and are
   compared with the cached Stage-2 / 3 / 4 arrays and the stored BEST predictions. If it fails, stop: do not run cell 4.
4. **The IDRiD test, once.** Refuses without the confirmation token, without a passed gate from this runtime, or if the lock
   exists. The 100-image primary result is written first, then the 103-image sensitivity analysis.
5. Print the reports.

GPU runtime (T4). No lesion / vessel shuffles, no 50 / 50 fusion and no other rule are computed on IDRiD.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import idrid_grading_eval as ig
import torch
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] PROTOCOL + THE EIGHT FROZEN MODEL FILES ====
PROTOCOL, PROTOCOL_SHA = ig.load_protocol()
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
assert torch.cuda.is_available(), "GPU runtime required (the protocol's settings are the T4 settings of the APTOS runs)"
PATHS = ig.verify_checkpoints(PROTOCOL, DRIVE_ROOT)        # size + SHA-256 of every file; BEST pointers unchanged
print("protocol", PROTOCOL_SHA, "| commit", REPO_COMMIT, "| GPU", torch.cuda.get_device_name(0))
for _name, _entry in PROTOCOL["checkpoints"].items():
    print(f"  {_name:<20} {_entry['sha256']}  {_entry.get('best_slot', '')}")
print("primary exclusions:", PROTOCOL["dataset"]["primary_exclusions"])
print("fusion:", {k: PROTOCOL["fusion"][k] for k in ("fused_0", "fused_1", "fused_2", "fused_3", "decode")})
print("inference:", PROTOCOL["inference"])

In [ ]:
# ==== [3] APTOS PARITY GATE -- must pass before any IDRiD image is read ====
GEN4 = cache.stage4_generation_id(PROTOCOL["checkpoints"]["stage4_unet"]["sha256"], len(v2cfg.STAGE4_V2A_CLASSES))
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=PROTOCOL["checkpoints"]["stage4_unet"]["sha256"])   # cached arrays, read from Drive
OUT_ROOT = posixpath.join(EXP, "IDRiDGrading")
PARITY = ig.run_parity(DRIVE_ROOT, posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images"), EXP,
                       posixpath.join(OUT_ROOT, f"parity_{PROTOCOL_SHA[:12]}"), bundle=BUNDLE,
                       aptos_processed_dir=posixpath.join(colab_config.APTOS2019_PROCESSED_DIR, "train_images"))
print(json.dumps({"PASS": PARITY["PASS"], "failures": PARITY["failures"], "prediction_checks": PARITY["prediction_checks"],
                  "environment": PARITY["environment"]}, indent=1))
assert PARITY["PASS"], "PARITY FAILED -- stop here. Do not run the IDRiD cell. Report the failures."


In [ ]:
# ==== [4] THE ONE-TIME IDRiD GRADING TEST ====
# Runs once. To run it, set CONFIRM to the token printed below. A second run is refused; after a technical failure only,
# set TECHNICAL_RERUN_REASON to a written reason (it is recorded with the run).
print("token:", ig.CONFIRMATION_TOKEN)
CONFIRM = None
TECHNICAL_RERUN_REASON = None
RESULT = ig.run_idrid(DRIVE_ROOT, colab_config.IDRID_GRADING_RAW_DIR, OUT_ROOT, PARITY, confirm=CONFIRM,
                      technical_rerun_reason=TECHNICAL_RERUN_REASON,
                      command="colab/notebooks/idrid_grading_evaluation.ipynb cell 4")

In [ ]:
# ==== [5] REPORTS (reads what cell 4 wrote) ====
print(open(posixpath.join(RESULT["out_dir"], "primary_report_100.md"), encoding="utf-8").read())
print(open(posixpath.join(RESULT["out_dir"], "sensitivity_report_103.md"), encoding="utf-8").read())
print("output:", RESULT["out_dir"])